#Implement Your Own Retriever

In [1]:
import re

docs = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines information retrieval and text generation.",
    "LangChain can be used to build RAG pipelines easily.",
    "Transformers use self-attention to handle sequential data."
]

query = "What is RAG?"

def retrieve_context(query, docs, k=1):
    # Normalize and split query into a set of unique words
    query_words = set(re.sub(r'[^\w\s]', '', query.lower()).split())

    scored_docs = []
    for doc in docs:
        # Normalize and split document words
        doc_words = re.sub(r'[^\w\s]', '', doc.lower()).split()
        # Count overlaps
        score = sum(1 for word in doc_words if word in query_words)
        scored_docs.append((score, doc))

    # Sort by score in descending order
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    # Return top-k document strings
    return [doc for score, doc in scored_docs[:k]]

context = retrieve_context(query, docs)
print("Retrieved context:", context)

Retrieved context: ['RAG stands for Retrieval-Augmented Generation.']


# Write the Prompt Template

In [2]:
def make_prompt(context, question):
    # Convert list of context documents into a clean string block
    context_str = " ".join(context) if isinstance(context, list) else context
    return f"Context: {context_str}\nQuestion: {question}\nAnswer:"

prompt_text = make_prompt(context, query)
print(prompt_text)

Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer:


# Plug in a HuggingFace Model

In [4]:
from transformers import pipeline

# Initialize the pipeline
# pad_token_id is explicitly set to clear the default generation warnings
hf_gen = pipeline(
    task="text-generation",
    model="distilgpt2",
    max_new_tokens=50,
    temperature=0.8,
    do_sample=True,
    pad_token_id=50256
)

response = hf_gen(prompt_text)[0]["generated_text"]
print(response)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer: RAG does not support Retrieval-Augmented Generation.
Question: Do you think RAG means Retrieval-Augmented Generation?
Answer: RAG is not specifically defined as Retrieval-Augmented Generation, but it


# Connect It All: Build a Mini RAG Function

In [5]:
def mini_rag(query):
    context = retrieve_context(query, docs)
    prompt = make_prompt(context, query)
    result = hf_gen(prompt)[0]["generated_text"]
    return result

# Test the mini RAG pipeline
print(mini_rag("How do transformers work?"))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: Transformers use self-attention to handle sequential data.
Question: How do transformers work?
Answer: The first thing you need to do is add a self-attention call. It is called a transformer. It is a component of a constructor that uses a trait named . The transformer is a constructor that uses a method that returns a list
